# DeepSeek-OCR batch OCR on Colab T4 with Google Drive checkpoints

This notebook installs vLLM, runs `deepseek-ai/DeepSeek-OCR` on PDF pages rendered from `/download/kasus anak/pdf`, and checkpoints progress after every page into a dedicated Google Drive folder. If the Colab runtime disconnects or GPU time runs out, rerun the notebook later; completed pages are skipped when their checkpoint and output file exist.

Primary references used while preparing this notebook:
- DeepSeek-OCR model card: https://huggingface.co/deepseek-ai/DeepSeek-OCR
- DeepSeek-OCR repository: https://github.com/deepseek-ai/DeepSeek-OCR
- vLLM DeepSeek-OCR recipe: https://docs.vllm.ai/projects/recipes/en/latest/DeepSeek/DeepSeek-OCR.html


## 1. Install runtime dependencies

Run this once per fresh Colab runtime. The first pass installs a matched PyTorch `2.9.0` CUDA 12.9 + vLLM `0.12.0` stack and intentionally restarts the Colab runtime once. After the restart, rerun this cell and continue with the notebook.

In [ ]:
import importlib
import json
import os
import subprocess
import sys
from pathlib import Path

# Colab T4 currently exposes a CUDA 12 runtime. Keep PyTorch and vLLM on a
# matched CUDA/PyTorch ABI stack; otherwise vLLM can fail with errors such as
# libcudart.so.13 missing or vllm/_C.abi3.so undefined symbol.
VLLM_VERSION = "0.12.0"
TORCH_VERSION = "2.9.0"
TORCHVISION_VERSION = "0.24.0"
TORCHAUDIO_VERSION = "2.9.0"
PYTORCH_CUDA_INDEX = "https://download.pytorch.org/whl/cu129"
ENV_MARKER = Path("/content/.deepseek_ocr_vllm_env_ready.json")
ENV_SPEC = {
    "vllm": VLLM_VERSION,
    "torch": TORCH_VERSION,
    "torchvision": TORCHVISION_VERSION,
    "torchaudio": TORCHAUDIO_VERSION,
    "cuda_index": PYTORCH_CUDA_INDEX,
}

def run_cmd(*args):
    cmd = [sys.executable, "-m", *args]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)
    importlib.invalidate_caches()

def marker_matches() -> bool:
    if not ENV_MARKER.exists():
        return False
    try:
        return json.loads(ENV_MARKER.read_text()) == ENV_SPEC
    except Exception:
        return False

if not marker_matches():
    # Remove incompatible leftovers from previous attempts. Missing packages are fine.
    run_cmd("pip", "uninstall", "-y", "vllm", "vllm-nightly", "torch", "torchvision", "torchaudio")

    # Install PyTorch first from the CUDA 12.9 index, then vLLM against that ABI.
    run_cmd(
        "pip", "install", "-q", "--no-cache-dir", "-U",
        f"torch=={TORCH_VERSION}",
        f"torchvision=={TORCHVISION_VERSION}",
        f"torchaudio=={TORCHAUDIO_VERSION}",
        "--index-url", PYTORCH_CUDA_INDEX,
    )
    run_cmd(
        "pip", "install", "-q", "--no-cache-dir", "-U",
        "pymupdf", "pillow", "tqdm", "pandas",
        f"vllm=={VLLM_VERSION}",
        "--extra-index-url", PYTORCH_CUDA_INDEX,
    )

    ENV_MARKER.write_text(json.dumps(ENV_SPEC, indent=2, sort_keys=True))
    print("Installed matched PyTorch/vLLM CUDA stack. Restarting this Colab runtime once; after it restarts, rerun from this cell.")
    os.kill(os.getpid(), 9)

# If a previous failed import left Python modules around, remove the Python-level
# entries before importing. Native libraries still require the one-time restart above.
for module_name in list(sys.modules):
    if module_name == "vllm" or module_name.startswith("vllm."):
        del sys.modules[module_name]

import torch
from vllm.model_executor.models.deepseek_ocr import NGramPerReqLogitsProcessor
print("Torch:", torch.__version__, "CUDA runtime:", torch.version.cuda)
print(f"DeepSeek-OCR vLLM integration is available with vLLM {VLLM_VERSION}.")


## 2. Mount Google Drive and configure paths

The checkpoint folder is intentionally separate from the input folder. Keep `CHECKPOINT_ROOT` on Google Drive so progress survives Colab runtime resets.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

MODEL_ID = "deepseek-ai/DeepSeek-OCR"

# User-requested input path. If you store the same folder in Drive instead,
# the fallback below will use /content/drive/MyDrive/download/kasus anak/pdf.
INPUT_PDF_DIR = Path("/download/kasus anak/pdf")
DRIVE_INPUT_FALLBACK = Path("/content/drive/MyDrive/download/kasus anak/pdf")
if not INPUT_PDF_DIR.exists() and DRIVE_INPUT_FALLBACK.exists():
    INPUT_PDF_DIR = DRIVE_INPUT_FALLBACK

# Dedicated checkpoint and output folder on Google Drive.
CHECKPOINT_ROOT = Path("/content/drive/MyDrive/deepseek_ocr_kasus_anak_checkpoint")
PAGE_IMAGE_ROOT = CHECKPOINT_ROOT / "page_images"
PAGE_OUTPUT_ROOT = CHECKPOINT_ROOT / "page_outputs"
COMBINED_OUTPUT_ROOT = CHECKPOINT_ROOT / "combined_pdf_outputs"
LOG_ROOT = CHECKPOINT_ROOT / "logs"
PROGRESS_JSONL = CHECKPOINT_ROOT / "progress.jsonl"
ERROR_JSONL = CHECKPOINT_ROOT / "errors.jsonl"
RUN_MANIFEST_JSON = CHECKPOINT_ROOT / "last_run_manifest.json"

for path in [CHECKPOINT_ROOT, PAGE_IMAGE_ROOT, PAGE_OUTPUT_ROOT, COMBINED_OUTPUT_ROOT, LOG_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("Input PDF folder:", INPUT_PDF_DIR)
print("Checkpoint folder:", CHECKPOINT_ROOT)
assert INPUT_PDF_DIR.exists(), f"Input folder does not exist: {INPUT_PDF_DIR}"


## 3. OCR settings

Defaults are conservative for a 16 GB NVIDIA T4: one page at a time, FP16, no prefix cache, and no multimodal processor cache. If you hit CUDA out-of-memory, lower `RENDER_DPI` to 150, lower `MAX_TOKENS`, or set `STOP_ON_ERROR = True` so the notebook stops at the first failing page.

In [ ]:
PROMPT = "<image>\n<|grounding|>Convert the document to markdown."

# Rendering and generation settings.
RENDER_DPI = 200
MAX_TOKENS = 4096
GPU_MEMORY_UTILIZATION = 0.90

# Resume controls.
MAX_PAGES_THIS_RUN = None       # Example: 100, or None for all remaining pages.
STOP_AFTER_MINUTES = None       # Example: 330 to stop before a 6-hour session ends.
STOP_ON_ERROR = False           # Failed pages are checkpointed and can be retried later.
RETRY_FAILED = True             # Reprocess pages with previous status == "failed".
OVERWRITE_DONE = False          # Keep False for normal resume behavior.

# Optional quick test: set to a small integer for the first run.
ONLY_FIRST_N_PDFS = None


## 4. Helpers for checkpointing, PDF rendering, and task discovery

In [ ]:
import hashlib
import json
import os
import re
import time
import traceback
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from typing import Dict, Iterable, List, Optional

import fitz
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

def now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

def safe_slug(value: str, max_len: int = 120) -> str:
    value = value.replace("\\", "/")
    value = re.sub(r"[^A-Za-z0-9._/-]+", "_", value).strip("._/")
    value = value.replace("/", "__")
    if len(value) > max_len:
        digest = hashlib.sha1(value.encode("utf-8")).hexdigest()[:12]
        value = value[: max_len - 13] + "_" + digest
    return value or "item"

def file_sha1(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha1()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    os.replace(tmp, path)

def atomic_write_json(path: Path, data) -> None:
    atomic_write_text(path, json.dumps(data, ensure_ascii=False, indent=2, sort_keys=True))

def append_jsonl(path: Path, event: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    line = json.dumps(event, ensure_ascii=False, sort_keys=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(line + "\n")
        f.flush()
        os.fsync(f.fileno())

def read_latest_progress(path: Path) -> Dict[str, dict]:
    latest = {}
    if not path.exists():
        return latest
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                event = json.loads(line)
            except json.JSONDecodeError:
                print(f"Skipping malformed progress line {line_no}")
                continue
            task_id = event.get("task_id")
            if task_id:
                latest[task_id] = event
    return latest

@dataclass(frozen=True)
class PageTask:
    task_id: str
    pdf_path: str
    pdf_rel_path: str
    pdf_sha1: str
    page_index: int
    page_number: int
    page_count: int
    image_path: str
    output_md_path: str

def render_pdf_page(task: PageTask, dpi: int = RENDER_DPI) -> Path:
    image_path = Path(task.image_path)
    if image_path.exists() and image_path.stat().st_size > 0:
        return image_path
    image_path.parent.mkdir(parents=True, exist_ok=True)
    with fitz.open(task.pdf_path) as doc:
        page = doc.load_page(task.page_index)
        zoom = dpi / 72.0
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
        tmp = image_path.with_name(image_path.stem + ".tmp" + image_path.suffix)
        pix.save(str(tmp))
    os.replace(tmp, image_path)
    return image_path

def discover_pdf_tasks(input_dir: Path) -> List[PageTask]:
    pdfs = sorted(input_dir.rglob("*.pdf"))
    if ONLY_FIRST_N_PDFS is not None:
        pdfs = pdfs[:ONLY_FIRST_N_PDFS]
    tasks: List[PageTask] = []
    for pdf_path in tqdm(pdfs, desc="Scanning PDFs"):
        rel = pdf_path.relative_to(input_dir).as_posix()
        pdf_slug = safe_slug(rel)
        digest = file_sha1(pdf_path)
        with fitz.open(pdf_path) as doc:
            page_count = doc.page_count
        for page_index in range(page_count):
            page_number = page_index + 1
            task_id = hashlib.sha1(f"{digest}:{rel}:{page_number}".encode("utf-8")).hexdigest()
            page_name = f"{pdf_slug}__page_{page_number:05d}"
            tasks.append(PageTask(
                task_id=task_id,
                pdf_path=str(pdf_path),
                pdf_rel_path=rel,
                pdf_sha1=digest,
                page_index=page_index,
                page_number=page_number,
                page_count=page_count,
                image_path=str(PAGE_IMAGE_ROOT / f"{page_name}.png"),
                output_md_path=str(PAGE_OUTPUT_ROOT / f"{page_name}.md"),
            ))
    return tasks

def is_task_done(task: PageTask, progress: Dict[str, dict]) -> bool:
    if OVERWRITE_DONE:
        return False
    event = progress.get(task.task_id)
    if not event or event.get("status") != "done":
        return False
    output_path = Path(event.get("output_md_path", task.output_md_path))
    return output_path.exists() and output_path.stat().st_size > 0

def should_run_task(task: PageTask, progress: Dict[str, dict]) -> bool:
    if is_task_done(task, progress):
        return False
    event = progress.get(task.task_id)
    if event and event.get("status") == "failed" and not RETRY_FAILED:
        return False
    return True


## 5. Build the task manifest and check resume state

In [ ]:
tasks = discover_pdf_tasks(INPUT_PDF_DIR)
progress = read_latest_progress(PROGRESS_JSONL)
pending_tasks = [task for task in tasks if should_run_task(task, progress)]
done_count = sum(1 for task in tasks if is_task_done(task, progress))

manifest = {
    "created_at": now_iso(),
    "model_id": MODEL_ID,
    "input_pdf_dir": str(INPUT_PDF_DIR),
    "checkpoint_root": str(CHECKPOINT_ROOT),
    "total_pages": len(tasks),
    "done_pages": done_count,
    "pending_pages": len(pending_tasks),
    "render_dpi": RENDER_DPI,
    "max_tokens": MAX_TOKENS,
}
atomic_write_json(RUN_MANIFEST_JSON, manifest)

print(json.dumps(manifest, indent=2))
pd.DataFrame([asdict(task) for task in tasks]).to_csv(CHECKPOINT_ROOT / "all_page_tasks.csv", index=False)
pd.DataFrame([asdict(task) for task in pending_tasks]).to_csv(CHECKPOINT_ROOT / "pending_page_tasks.csv", index=False)


## 6. Load DeepSeek-OCR with vLLM

This loads the model once. On a T4, the first load can take several minutes because Colab must download model weights and initialize CUDA kernels.

In [ ]:
import gc
from vllm import LLM, SamplingParams
from vllm.model_executor.models.deepseek_ocr import NGramPerReqLogitsProcessor

# Do not call torch.cuda.* before LLM(...). In notebooks that initializes CUDA
# too early and forces vLLM to use slower spawn multiprocessing on Colab.
llm = LLM(
    model=MODEL_ID,
    dtype="float16",
    trust_remote_code=True,
    enable_prefix_caching=False,
    mm_processor_cache_gb=0,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    max_model_len=8192,
    limit_mm_per_prompt={"image": 1},
    logits_processors=[NGramPerReqLogitsProcessor],
)

import torch
print("GPU:", torch.cuda.get_device_name(0))

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_TOKENS,
    skip_special_tokens=False,
    extra_args={
        "ngram_size": 30,
        "window_size": 90,
        "whitelist_token_ids": {128821, 128822},
    },
)


## 7. Run OCR with page-level checkpointing

Each page writes one markdown file and then appends a `done` event to `progress.jsonl` on Google Drive. Re-running this cell skips completed pages.

In [ ]:
def run_ocr_for_task(task: PageTask) -> str:
    image_path = render_pdf_page(task, dpi=RENDER_DPI)
    image = Image.open(image_path).convert("RGB")
    model_input = [{"prompt": PROMPT, "multi_modal_data": {"image": image}}]
    outputs = llm.generate(model_input, sampling_params)
    return outputs[0].outputs[0].text

progress = read_latest_progress(PROGRESS_JSONL)
run_queue = [task for task in tasks if should_run_task(task, progress)]
if MAX_PAGES_THIS_RUN is not None:
    run_queue = run_queue[:MAX_PAGES_THIS_RUN]

print(f"Pages queued this run: {len(run_queue)}")
started_at = time.time()
processed = 0

for task in tqdm(run_queue, desc="OCR pages"):
    if STOP_AFTER_MINUTES is not None:
        elapsed_minutes = (time.time() - started_at) / 60.0
        if elapsed_minutes >= STOP_AFTER_MINUTES:
            print(f"Stopping because STOP_AFTER_MINUTES={STOP_AFTER_MINUTES} was reached.")
            break

    current_progress = read_latest_progress(PROGRESS_JSONL)
    if is_task_done(task, current_progress):
        continue

    start_event = {
        **asdict(task),
        "status": "running",
        "updated_at": now_iso(),
        "render_dpi": RENDER_DPI,
        "max_tokens": MAX_TOKENS,
    }
    append_jsonl(PROGRESS_JSONL, start_event)

    try:
        text = run_ocr_for_task(task)
        header = (
            f"<!-- source_pdf: {task.pdf_rel_path} -->\n"
            f"<!-- page: {task.page_number}/{task.page_count} -->\n"
            f"<!-- task_id: {task.task_id} -->\n\n"
        )
        atomic_write_text(Path(task.output_md_path), header + text.strip() + "\n")
        done_event = {
            **asdict(task),
            "status": "done",
            "updated_at": now_iso(),
            "output_bytes": Path(task.output_md_path).stat().st_size,
        }
        append_jsonl(PROGRESS_JSONL, done_event)
        processed += 1
    except Exception as exc:
        error_event = {
            **asdict(task),
            "status": "failed",
            "updated_at": now_iso(),
            "error_type": type(exc).__name__,
            "error": str(exc),
            "traceback": traceback.format_exc(limit=10),
        }
        append_jsonl(PROGRESS_JSONL, error_event)
        append_jsonl(ERROR_JSONL, error_event)
        print(f"Failed {task.pdf_rel_path} page {task.page_number}: {type(exc).__name__}: {exc}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        if STOP_ON_ERROR:
            raise

print(f"Processed pages this run: {processed}")


## 8. Combine page outputs back into one markdown file per PDF

Run this after any OCR session. It only includes pages whose markdown output exists.

In [ ]:
from collections import defaultdict

progress = read_latest_progress(PROGRESS_JSONL)
tasks_by_pdf = defaultdict(list)
for task in tasks:
    tasks_by_pdf[task.pdf_rel_path].append(task)

summary_rows = []
for pdf_rel_path, pdf_tasks in sorted(tasks_by_pdf.items()):
    pdf_tasks = sorted(pdf_tasks, key=lambda t: t.page_number)
    completed_parts = []
    completed = 0
    for task in pdf_tasks:
        output_path = Path(task.output_md_path)
        if output_path.exists() and output_path.stat().st_size > 0:
            completed += 1
            completed_parts.append(output_path.read_text(encoding="utf-8"))
        else:
            completed_parts.append(f"\n\n<!-- Missing OCR output for page {task.page_number} -->\n")

    combined_name = safe_slug(pdf_rel_path) + ".md"
    combined_path = COMBINED_OUTPUT_ROOT / combined_name
    combined_text = "\n\n---\n\n".join(completed_parts).strip() + "\n"
    atomic_write_text(combined_path, combined_text)
    summary_rows.append({
        "pdf_rel_path": pdf_rel_path,
        "pages_total": len(pdf_tasks),
        "pages_completed": completed,
        "combined_output": str(combined_path),
    })

summary_df = pd.DataFrame(summary_rows)
summary_csv = CHECKPOINT_ROOT / "combined_output_summary.csv"
summary_df.to_csv(summary_csv, index=False)
display(summary_df.sort_values(["pages_completed", "pages_total"], ascending=[True, False]).head(20))
print("Summary CSV:", summary_csv)


## 9. Resume checklist

For the next Colab session:
1. Use a GPU runtime.
2. Run sections 1 through 7 again.
3. Keep the same `CHECKPOINT_ROOT` on Google Drive.
4. The notebook will read `progress.jsonl`, verify page markdown files, and skip completed pages.
5. Run section 8 to regenerate combined per-PDF markdown outputs.